In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import optuna
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score
from sklearn.model_selection import KFold

In [3]:
# Load your data
data = pd.read_csv('QGIS_df.csv')

In [4]:
house_df = data[data['Category'] == 'house']
condo_df = data[data['Category'] == 'condo']
apt_df = data[data['Category'] == 'apartment']
com_df = data[data['Category'] == 'commercial']
land_df = data[data['Category'] == 'land']
RR_df = pd.concat([house_df, condo_df, apt_df], ignore_index=True)
CR_df = pd.concat([com_df, land_df], ignore_index=True)

In [7]:
# Cross-validation setup
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

In [24]:
def objective(trial):
    # Hyperparameter ranges as specified
    params = {
        'n_neighbors': trial.suggest_int('n_neighbors', 3, 50),
        'weights': trial.suggest_categorical('weights', ['uniform', 'distance']),
        'algorithm': trial.suggest_categorical('algorithm', ['ball_tree', 'kd_tree', 'brute', 'auto']),
        'leaf_size': trial.suggest_int('leaf_size', 10, 50),
        'p': trial.suggest_int('p', 1, 2)
    }
    
    # Create KNN Regressor with current hyperparameters
    knn_regressor = KNeighborsRegressor(
        n_neighbors=params['n_neighbors'],
        weights=params['weights'],
        algorithm=params['algorithm'],
        leaf_size=params['leaf_size'],
        p=params['p']
    )
    
    # Scale the features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Perform cross-validation
    scores = cross_val_score(knn_regressor, X_scaled, y, cv=5, scoring='r2')
    #scores = cross_val_score(et_regressor, X, y, cv=kfold, scoring='r2')
    #return np.mean(-scores)
    return np.mean(scores)

In [26]:
def optimize_knn(X, y):
    # Create a study object and optimize the objective function
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=100)
    
    # Print best parameters and score
    print("Best trial:")
    trial = study.best_trial
    print(f"  Value: {trial.value}")
    print("  Params: ")
    for key, value in trial.params.items():
        print(f"    {key}: {value}")
    
    return study.best_params

In [28]:
# Train Extra Trees with Best Params and Get Feature Importances
def get_feature_importance(X, y, best_params):
    model = KNeighborsRegressor(**best_params, random_state=42)
    model.fit(X, y)
    
    # Extract feature importances
    feature_importances = model.feature_importances_
    
    # Convert to DataFrame
    importance_df = pd.DataFrame({'Feature': X.columns, 'Importance': feature_importances})
    importance_df = importance_df.sort_values(by='Importance', ascending=False)
    
    # Print Top Features
    print("\nTop Features:")
    print(importance_df)
    
    # Plot Feature Importances
    plt.figure(figsize=(10, 6))
    plt.barh(importance_df['Feature'], importance_df['Importance'], color='royalblue')
    plt.xlabel('Importance')
    plt.ylabel('Feature')
    plt.title('Feature Importance in KNN')
    plt.gca().invert_yaxis()  # Highest importance at the top
    plt.show()
    
    return importance_df

In [30]:
# Assuming df is already loaded and preprocessed
y = house_df['Price']
X = house_df[["LRTHubDist", 'Num_Bedrooms', 'Num_Bathrooms', 'Floor_Area', 'Land_Area', 
        'BusCount', 'SchoolCount', 'MallCount', 'HospitalCount']]

# Optimize and get best parameters
best_params = optimize_knn(X, y)
feature_importances = get_feature_importance(X, y, best_params)

# Train final model with best parameters
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
final_model = KNeighborsRegressor(**best_params)
final_model.fit(X_scaled, y)

[I 2025-02-03 19:32:13,097] A new study created in memory with name: no-name-0bbb4342-7ff6-4488-ba62-85791eb6e224
[I 2025-02-03 19:32:23,296] Trial 0 finished with value: -2.2334088137617867 and parameters: {'n_neighbors': 22, 'weights': 'uniform', 'algorithm': 'auto', 'leaf_size': 17, 'p': 2}. Best is trial 0 with value: -2.2334088137617867.
[I 2025-02-03 19:32:54,003] Trial 1 finished with value: -2.0211171946221684 and parameters: {'n_neighbors': 36, 'weights': 'distance', 'algorithm': 'ball_tree', 'leaf_size': 37, 'p': 2}. Best is trial 1 with value: -2.0211171946221684.
[I 2025-02-03 19:33:27,452] Trial 2 finished with value: -1.548487825263287 and parameters: {'n_neighbors': 44, 'weights': 'distance', 'algorithm': 'ball_tree', 'leaf_size': 49, 'p': 2}. Best is trial 2 with value: -1.548487825263287.
[I 2025-02-03 19:33:47,756] Trial 3 finished with value: -0.7985680849309096 and parameters: {'n_neighbors': 49, 'weights': 'distance', 'algorithm': 'auto', 'leaf_size': 38, 'p': 1}. 

Best trial:
  Value: -0.7523720510734117
  Params: 
    n_neighbors: 14
    weights: distance
    algorithm: ball_tree
    leaf_size: 38
    p: 1


TypeError: KNeighborsRegressor.__init__() got an unexpected keyword argument 'random_state'

In [ ]:
# Assuming df is already loaded and preprocessed
y = condo_df['Price']
X = condo_df[["LRTHubDist", 'Num_Bedrooms', 'Num_Bathrooms', 'Floor_Area', 'Land_Area', 
        'BusCount', 'SchoolCount', 'MallCount', 'HospitalCount']]

# Optimize and get best parameters
best_params = optimize_knn(X, y)
feature_importances = get_feature_importance(X, y, best_params)

# Train final model with best parameters
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
final_model = KNeighborsRegressor(**best_params)
final_model.fit(X_scaled, y)

[I 2025-02-03 19:37:06,708] A new study created in memory with name: no-name-37d987b7-c3e2-413e-ae12-959ace9d593c
[I 2025-02-03 19:37:17,861] Trial 0 finished with value: 0.24210914476612588 and parameters: {'n_neighbors': 40, 'weights': 'distance', 'algorithm': 'kd_tree', 'leaf_size': 26, 'p': 1}. Best is trial 0 with value: 0.24210914476612588.
C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:1011: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\metrics\_scorer.py", line 137, in __call__
    score = scorer._score(
            ^^^^^^^^^^^^^^
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\metrics\_scorer.py", line 345, in _score
    y_pred = method_caller(
             ^^^^^^^^^^^^^^
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\sklearn\metrics\_s